# Computational Overhead Estimates

This revision notebook measures bounded timing probes and extrapolates to the paper-scale settings. It does **not** run the full experiments and it does **not** save models, samples, tables, or figures.

The proportional estimate is

$$
T_{\mathrm{full}} \approx T_{\mathrm{probe}}\,\frac{W_{\mathrm{full}}}{W_{\mathrm{probe}}},
$$

where $W$ is an explicit work count: epochs for fixed-size toy routines, field-passes for SRCNN downscaling, optimizer sample-passes for Flow Matching training, and generated-sample ODE-step passes for Flow Matching sampling. Peak VRAM is measured during the probe because memory depends primarily on model, tensor shape, and batch size rather than on the number of epochs.

## Setup And Hardware

This cell imports the project helpers, records the tested hardware, and blocks accidental calls to `torch.save` and `np.save`. The final summary reports whether any save call was attempted.

In [1]:

import gc
import math
import os
import platform
import sys
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'experiments').is_dir():
            return candidate
    return start
CODE_DIR = Path(os.environ['ETA_CODE_DIR']) if 'ETA_CODE_DIR' in os.environ else _find_repo_root(Path.cwd())
WORK_DIR = Path(os.environ.get('ETA_WORK_DIR', CODE_DIR))
SRC_DIR = CODE_DIR / 'src'
for import_path in (CODE_DIR, SRC_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from models import FCNN, SRCNN, UNet
from revision_utils import format_seconds, hardware_summary, measure_wall_time, train_precip_eta_no_save, train_srcnn_mse_no_save
from train_utils import generate_2d_gaussian, grf_pretrain, train_eta_xuy, train_eta_xy, train_nn
from utils import global_seed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GB = 1024 ** 3

SAVE_BLOCK_EVENTS = []
_ORIGINAL_TORCH_SAVE = torch.save
_ORIGINAL_NP_SAVE = np.save

def _blocked_torch_save(obj, f, *args, **kwargs):
    SAVE_BLOCK_EVENTS.append({'api': 'torch.save', 'target': str(f)})
    return None

def _blocked_np_save(file, arr, *args, **kwargs):
    SAVE_BLOCK_EVENTS.append({'api': 'np.save', 'target': str(file)})
    return None

torch.save = _blocked_torch_save
np.save = _blocked_np_save

def system_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().total / GB
    except Exception:
        try:
            with open('/proc/meminfo', 'r', encoding='utf-8') as handle:
                for line in handle:
                    if line.startswith('MemTotal:'):
                        return float(line.split()[1]) * 1024 / GB
        except Exception:
            return np.nan
    return np.nan

def process_rss_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / GB
    except Exception:
        return np.nan

hardware = hardware_summary(DEVICE)
hardware['system_ram_GB'] = system_ram_gb()
if torch.cuda.is_available():
    try:
        free_vram, total_vram = torch.cuda.mem_get_info(DEVICE)
        hardware['gpu_vram_free_at_start_GB'] = free_vram / GB
        hardware['gpu_vram_total_from_runtime_GB'] = total_vram / GB
    except Exception:
        hardware['gpu_vram_free_at_start_GB'] = np.nan
        hardware['gpu_vram_total_from_runtime_GB'] = hardware.get('gpu_total_vram_GB', np.nan)
else:
    hardware['gpu_vram_free_at_start_GB'] = np.nan
    hardware['gpu_vram_total_from_runtime_GB'] = np.nan

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 200)
display(pd.DataFrame([hardware]))

runtime_rows = []


def maybe_data_parallel(model):
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        return nn.DataParallel(model)
    return model


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def as_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def measure_estimate(
    experiment,
    component,
    func,
    scale_factor,
    paper_setting,
    timed_probe,
    *args,
    notes='',
    **kwargs,
):
    cleanup()
    rss_before = process_rss_gb()
    result = None
    status = 'ok'
    error = ''
    seconds = np.nan
    peak = {'peak_vram_allocated_GB': np.nan, 'peak_vram_reserved_GB': np.nan}
    try:
        result, seconds, peak = measure_wall_time(func, *args, device=DEVICE, **kwargs)
    except RuntimeError as exc:
        status = 'failed'
        error = f'{type(exc).__name__}: {str(exc).splitlines()[0]}'
    except Exception as exc:
        status = 'failed'
        error = f'{type(exc).__name__}: {str(exc).splitlines()[0]}'
    rss_after = process_rss_gb()
    estimated_seconds = as_float(seconds) * as_float(scale_factor) if np.isfinite(as_float(seconds)) else np.nan
    runtime_rows.append({
        'experiment': experiment,
        'component': component,
        'status': status,
        'paper_setting': paper_setting,
        'timed_probe': timed_probe,
        'probe_seconds': as_float(seconds),
        'probe_time': format_seconds(seconds) if np.isfinite(as_float(seconds)) else np.nan,
        'scale_factor': as_float(scale_factor),
        'estimated_full_seconds': estimated_seconds,
        'estimated_full_time': format_seconds(estimated_seconds) if np.isfinite(estimated_seconds) else np.nan,
        'peak_vram_allocated_GB': peak.get('peak_vram_allocated_GB', np.nan),
        'peak_vram_reserved_GB': peak.get('peak_vram_reserved_GB', np.nan),
        'process_rss_before_GB': rss_before,
        'process_rss_after_GB': rss_after,
        'device': str(DEVICE),
        'notes': notes if status == 'ok' else f'{notes} ERROR: {error}',
    })
    cleanup()
    return result



def warm_up_device():
    if not torch.cuda.is_available():
        return
    warm_model = FCNN([16], activation='psilu', indim=2, outdim=1, init='xavier normal', positive_output=False).to(DEVICE)
    warm_x = torch.randn(64, 2, device=DEVICE)
    warm_y = warm_model(warm_x).sum()
    warm_y.backward()
    torch.cuda.synchronize(DEVICE)
    del warm_model, warm_x, warm_y
    cleanup()

warm_up_device()

print('CODE_DIR =', CODE_DIR)
print('WORK_DIR =', WORK_DIR)
print('DEVICE =', DEVICE)


,device_used,gpu_name,gpu_total_vram_GB,cuda_available,torch_version,cuda_version,python_version,os,cpu_model,system_ram_GB,gpu_vram_free_at_start_GB,gpu_vram_total_from_runtime_GB
0,cuda,Tesla V100-SXM2-32GB-LS,31.732544,True,2.2.0+cu118,11.8,3.11.8,Linux-4.18.0-477.10.1.el8_8.x86_64-x86_64-with-glibc2.28,x86_64,503.763378,NaN,31.732544


CODE_DIR = /path/to/eta
WORK_DIR = /path/to/eta
DEVICE = cuda


## Shared Data Preparation

This cell loads the toy data and the ERA5-Land precipitation fields used by the timing probes. The ERA5-Land trimming follows the main downscaling notebooks: HR daily maxima are trimmed above 240 mm, LR inputs are factor-10 subsamples, and the supervised downscaling split uses 0.5 years ($n=180$) of paired data.

In [2]:

import xarray as xr

# Toy data used by notebooks/toy--2D->1D.ipynb and notebooks/toy--2D->2D.ipynb.
toy_n_train = 100
toy_var_in = 10
toy_data_path = WORK_DIR / 'data/toy.pth'
if not toy_data_path.exists():
    raise FileNotFoundError(f'Missing toy data: {toy_data_path}')
try:
    TOY_DATA = torch.load(toy_data_path, map_location='cpu', weights_only=False)
except TypeError:
    TOY_DATA = torch.load(toy_data_path, map_location='cpu')

X_toy = TOY_DATA['X_all'].float()
Y_toy = TOY_DATA['Y_all'].float()
GRIDS_toy = TOY_DATA['in_grid'].float()
x_train_toy = TOY_DATA['x_train'].float()
y_train_toy = TOY_DATA['y_train'].float()
N_toy_grid = int(np.sqrt(GRIDS_toy.shape[0]))
U1_toy = Y_toy.reshape(-1, 1)

def fourier_mode_2d(x, freq=(1/6, 1/8), phase=(0, 0), amplitude=0.1):
    return -amplitude * torch.sin(2 * np.pi * freq[0] * x[..., 0] + phase[0]) * torch.sin(2 * np.pi * freq[1] * x[..., 1] + phase[1])

U2_toy = fourier_mode_2d(X_toy).reshape(-1, 1)
U_toy = torch.cat((U1_toy, U2_toy), dim=1)
u_train_toy = torch.cat((y_train_toy.reshape(-1, 1), fourier_mode_2d(x_train_toy).reshape(-1, 1)), dim=1)
U2_GRID_toy = fourier_mode_2d(GRIDS_toy).reshape(N_toy_grid, N_toy_grid)

def toy_observable(u):
    return 2 * torch.abs(u[:, 0]) + 0.5 * torch.abs(u[:, 1])

Y_toy_2d = toy_observable(U_toy)

# ERA5-Land data used by the precipitation, Flow Matching, and GEVD probes.
era5_path = WORK_DIR / 'data/era5land_USA_SouthEast_1999-2023_dailymax.nc'
if not era5_path.exists():
    raise FileNotFoundError(f'Missing ERA5-Land data: {era5_path}')

ds = xr.open_dataset(era5_path, engine='netcdf4')
tp_numpy = ds['tp'].values * 1000

ds_fact = 10
tp_ds_numpy = tp_numpy[:, ::ds_fact, ::ds_fact]
max_values_original = np.max(tp_numpy, axis=(1, 2))
sorted_indices_original = np.argsort(max_values_original)
sorted_max_values_original = max_values_original[sorted_indices_original]
trim_tail_thresh = 240
num_trim_days = len(sorted_max_values_original[sorted_max_values_original > trim_tail_thresh])
trim_days = sorted_indices_original[-num_trim_days:]
kept_days = np.delete(np.arange(len(tp_numpy)), trim_days)

tp_trim_numpy = tp_numpy[kept_days].astype(np.float32)
tp_trim_ds_numpy = tp_ds_numpy[kept_days].astype(np.float32)
max_values = np.max(tp_trim_numpy, axis=(1, 2))
sorted_indices = np.argsort(max_values)
sorted_max_values = max_values[sorted_indices]

num_years_supervised = 0.5
train_size = int((num_years_supervised / 25) * len(tp_trim_ds_numpy))
batch_size_srcnn = 64
train_input = torch.tensor(tp_trim_ds_numpy[:train_size], dtype=torch.float32).unsqueeze(1)
train_target = torch.tensor(tp_trim_numpy[:train_size], dtype=torch.float32).unsqueeze(1)
train_loader_srcnn = DataLoader(TensorDataset(train_input, train_target), batch_size=batch_size_srcnn, shuffle=True, num_workers=0)

# Probe the auxiliary/inference set with a bounded number of fields. Runtime is then scaled to all 9044 fields.
probe_aux_size = min(len(tp_trim_ds_numpy), 768)
probe_aux_indices = np.linspace(0, len(tp_trim_ds_numpy) - 1, probe_aux_size, dtype=int)
tp_probe_numpy = tp_trim_numpy[probe_aux_indices]
tp_probe_ds_numpy = tp_trim_ds_numpy[probe_aux_indices]
probe_test_batch_size = 256
probe_test_loader = DataLoader(
    TensorDataset(torch.tensor(tp_probe_ds_numpy, dtype=torch.float32).unsqueeze(1),
                  torch.tensor(tp_probe_numpy, dtype=torch.float32).unsqueeze(1)),
    batch_size=probe_test_batch_size,
    shuffle=False,
    num_workers=0,
)

setup_rows = [
    {'quantity': 'toy n_train', 'value': toy_n_train},
    {'quantity': 'toy auxiliary grid points', 'value': len(X_toy)},
    {'quantity': 'ERA5 trimmed fields', 'value': len(tp_trim_numpy)},
    {'quantity': 'ERA5 HR shape', 'value': tuple(tp_trim_numpy.shape[1:])},
    {'quantity': 'ERA5 LR shape', 'value': tuple(tp_trim_ds_numpy.shape[1:])},
    {'quantity': 'ERA5 supervised train fields', 'value': train_size},
    {'quantity': 'ERA5 probe auxiliary fields', 'value': probe_aux_size},
]
display(pd.DataFrame(setup_rows))


/tmp/ipykernel_47409/3301627061.py:40: FutureWarning: In a future version, xarray will not decode the variable 'step' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(era5_path, engine='netcdf4')


,quantity,value
0,toy n_train,100
1,toy auxiliary grid points,1000000
2,ERA5 trimmed fields,9044
3,ERA5 HR shape,"(80, 160)"
4,ERA5 LR shape,"(8, 16)"
5,ERA5 supervised train fields,180
6,ERA5 probe auxiliary fields,768


## Toy Example Timing Probes

The toy probes use the architecture and training schedule from `notebooks/toy--2D->1D.ipynb` and `notebooks/toy--2D->2D.ipynb`: a 3-hidden-layer, width-256 FCNN with `psilu`; MSE baseline for 3000 iterations; eta pretraining for 1000 iterations; eta continuation for 3000 iterations; $\lambda=1$; and $\omega=100$. Probe runs use two iterations and scale linearly by iteration count because each toy iteration uses the same supervised set and auxiliary grid as the full run.

In [3]:

toy_quantiles_1d = torch.cat((
    torch.linspace(0.0, 0.0001, 10), torch.linspace(0.0001, 0.001, 10), torch.linspace(0.001, 0.01, 10),
    torch.linspace(0.01, 0.1, 9), torch.linspace(0.1, 0.9, 20), torch.linspace(0.9, 0.99, 21),
    torch.linspace(0.99, 0.999, 21), torch.linspace(0.999, 0.9999, 21), torch.linspace(0.9999, 0.99999, 21),
    torch.linspace(0.99999, 0.999999, 21), torch.linspace(0.999999, 0.9999999, 21),
))
toy_quantiles_2d = torch.cat((
    torch.linspace(0, 0.9, 41), torch.linspace(0.9, 0.99, 21), torch.linspace(0.99, 0.999, 21),
    torch.linspace(0.999, 0.9999, 21), torch.linspace(0.9999, 0.99999, 21),
    torch.linspace(0.99999, 0.999999, 21), torch.linspace(0.999999, 0.9999999, 21),
))

toy_probe_iters = 10

def build_toy_fcnn(outdim, init='xavier normal', positive_output=False):
    model = FCNN([256, 256, 256], activation='psilu', indim=2, outdim=outdim, init=init, positive_output=positive_output)
    return maybe_data_parallel(model).to(DEVICE)

def run_toy_1d_mse(epochs):
    global_seed(1)
    model = build_toy_fcnn(outdim=1, init='xavier normal', positive_output=True)
    return train_nn(model, x_train_toy, y_train_toy, nn.MSELoss, torch.optim.Adam, epochs, toy_n_train, DEVICE, lr=1e-4)

def run_toy_2d_mse(epochs):
    global_seed(1)
    model = build_toy_fcnn(outdim=2, init='xavier normal', positive_output=False)
    return train_nn(model, x_train_toy, u_train_toy, nn.MSELoss, torch.optim.Adam, epochs, toy_n_train, DEVICE)

def run_toy_1d_pretrain(epochs):
    global_seed(25)
    model = build_toy_fcnn(outdim=1, init='xavier normal', positive_output=False)
    gaussian_field, _ = generate_2d_gaussian(seed=25, sigma=0.3)
    return grf_pretrain(gaussian_field, model, epochs, n_grid=50, grid_step=2, device=DEVICE)

def run_toy_2d_pretrain(epochs):
    global_seed(28)
    model = build_toy_fcnn(outdim=2, init='kaiming normal', positive_output=False)
    gaussian_field, _ = generate_2d_gaussian(seed=28, sigma=0.3)
    pretrain_field = torch.cat((gaussian_field.unsqueeze(-1), U2_GRID_toy.unsqueeze(-1)), dim=-1)
    return grf_pretrain(pretrain_field, model, epochs, n_grid=50, grid_step=2, device=DEVICE)

def initial_toy_1d_eta_model():
    pre = run_toy_1d_pretrain(1)
    return pre.model

def initial_toy_2d_eta_model():
    pre = run_toy_2d_pretrain(1)
    return pre.model

def run_toy_1d_eta(epochs):
    model = initial_toy_1d_eta_model()
    return train_eta_xy(model, x_train_toy, y_train_toy, X_toy, Y_toy, toy_quantiles_1d,
                        torch.optim.Adam, epochs, toy_n_train, DEVICE, _lamb=1.0, omega=100)

def run_toy_2d_eta(epochs):
    model = initial_toy_2d_eta_model()
    return train_eta_xuy(model, x_train_toy, u_train_toy, X_toy, Y_toy_2d, toy_observable, toy_quantiles_2d,
                         torch.optim.Adam, epochs, toy_n_train, DEVICE, _lamb=1.0, omega=100)

measure_estimate(
    'Toy 2D-to-1D', 'MSE baseline training', run_toy_1d_mse, 3000 / toy_probe_iters,
    'FCNN 256x3, psilu, Adam lr=1e-4, 3000 iterations, batch=100',
    f'{toy_probe_iters} iterations with full toy supervised set', toy_probe_iters,
    notes='Scale uses iteration count; no checkpoint saved.',
)
measure_estimate(
    'Toy 2D-to-1D', 'eta pretraining', run_toy_1d_pretrain, 1000 / toy_probe_iters,
    'Gaussian random-field pretrain, 1000 iterations, n_grid=50, grid_step=2',
    f'{toy_probe_iters} pretrain iterations', toy_probe_iters,
    notes='Probe uses same 50x50 pretraining grid as the notebook.',
)
measure_estimate(
    'Toy 2D-to-1D', 'eta continuation', run_toy_1d_eta, 3000 / toy_probe_iters,
    f'eta continuation, lambda=1, omega=100, {len(toy_quantiles_1d)} quantiles, 3000 iterations',
    f'{toy_probe_iters} eta iterations after a one-step warm-start pretrain', toy_probe_iters,
    notes='Scale uses iteration count with the full toy auxiliary grid.',
)
measure_estimate(
    'Toy 2D-to-2D', 'MSE baseline training', run_toy_2d_mse, 3000 / toy_probe_iters,
    'FCNN 256x3, psilu, Adam defaults, 3000 iterations, batch=100',
    f'{toy_probe_iters} iterations with full toy supervised set', toy_probe_iters,
    notes='Scale uses iteration count; no checkpoint saved.',
)
measure_estimate(
    'Toy 2D-to-2D', 'eta pretraining', run_toy_2d_pretrain, 1000 / toy_probe_iters,
    'Two-channel Gaussian/Fourier pretrain, 1000 iterations, n_grid=50, grid_step=2',
    f'{toy_probe_iters} pretrain iterations', toy_probe_iters,
    notes='Probe uses the same two-channel pretraining target as the notebook.',
)
measure_estimate(
    'Toy 2D-to-2D', 'eta continuation', run_toy_2d_eta, 3000 / toy_probe_iters,
    f'eta continuation, lambda=1, omega=100, {len(toy_quantiles_2d)} quantiles, 3000 iterations',
    f'{toy_probe_iters} eta iterations after a one-step warm-start pretrain', toy_probe_iters,
    notes='Scale uses iteration count with the full toy auxiliary grid.',
)


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:04<00:39,  4.40s/it]

 80%|████████  | 8/10 [00:04<00:00,  2.42it/s]

100%|██████████| 10/10 [00:04<00:00,  2.21it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

 30%|███       | 3/10 [00:00<00:00, 20.32it/s]

 60%|██████    | 6/10 [00:00<00:00, 20.76it/s]

 90%|█████████ | 9/10 [00:00<00:00, 19.83it/s]

100%|██████████| 10/10 [00:00<00:00, 20.01it/s]

## ERA5-Land Downscaling Timing Probes

The precipitation probes use the SRCNN setup from `notebooks/ERA5Land.ipynb`: `SRCNN(hidden_dim=64, num_blocks=3, scale_factor=10)`, supervised batch size 64, Adam with learning rate $3\times 10^{-4}$, 500 MSE epochs, and 150 eta epochs with empirical tail reference $g(u)=\max(u)$, $\lambda=1$, $\omega=30$, and roughly 102 tail days above 150 mm. The probe uses 768 auxiliary fields and scales by field-pass work to the full 9044-field auxiliary set.

In [4]:

def build_srcnn():
    return maybe_data_parallel(SRCNN(hidden_dim=64, num_blocks=3, scale_factor=ds_fact)).to(DEVICE)

def load_precip_mse_or_fresh():
    model = build_srcnn()
    ckpt = WORK_DIR / f'models/precip-srcnn/srcnn-mse-{num_years_supervised}yr-{ds_fact}ds.pth'
    source = 'fresh initialization'
    if ckpt.exists():
        try:
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state)
            source = f'loaded {ckpt.name}'
        except Exception as exc:
            source = f'fresh initialization; checkpoint load skipped ({type(exc).__name__})'
    return model, source

precip_mse_probe_epochs = 1
precip_eta_probe_epochs = 1
n_full_fields = len(tp_trim_numpy)

def mse_field_pass_work(num_epochs, n_aux):
    return num_epochs * (train_size + n_aux)

def eta_field_pass_work(num_epochs, n_aux, n_quantiles):
    return n_aux + num_epochs * (train_size + n_quantiles + n_aux)

def run_precip_mse_probe(num_epochs):
    global_seed(43)
    model = build_srcnn()
    return train_srcnn_mse_no_save(
        model, train_loader_srcnn, probe_test_loader, num_epochs, 3e-4,
        scheduler_step_size=200, scheduler_gamma=0.2, device=DEVICE,
    )

w1_tail_thresh = 150
num_w1_days = int(np.sum(sorted_max_values > w1_tail_thresh))
probe_sorted_indices = np.argsort(np.max(tp_probe_numpy, axis=(1, 2)))
w1_truedays_probe = probe_sorted_indices[-num_w1_days:]
w1_truemax = torch.tensor(sorted_max_values[-num_w1_days:], dtype=torch.float32)

mse_scale = mse_field_pass_work(500, n_full_fields) / mse_field_pass_work(precip_mse_probe_epochs, probe_aux_size)
measure_estimate(
    'ERA5-Land downscaling', 'MSE baseline training', run_precip_mse_probe, mse_scale,
    'SRCNN, 0.5-year supervised split, 500 epochs, batch=64, Adam lr=3e-4, StepLR(200, 0.2)',
    f'{precip_mse_probe_epochs} epoch over 180 supervised fields plus {probe_aux_size} auxiliary eval fields',
    precip_mse_probe_epochs,
    notes='Scale uses supervised plus auxiliary field-passes; no checkpoint saved.',
)

precip_eta_model, precip_eta_init_source = load_precip_mse_or_fresh()

def run_precip_eta_probe(num_epochs):
    model = deepcopy(precip_eta_model).to(DEVICE)
    return train_precip_eta_no_save(
        model, probe_test_loader, train_input, train_target, tp_probe_ds_numpy,
        w1_truemax, w1_truedays_probe, num_epochs, 3e-4, 1.0, 30, True,
        seed=43, device=DEVICE, keep_best_by_w1=True,
    )

eta_scale = eta_field_pass_work(150, n_full_fields, num_w1_days) / eta_field_pass_work(precip_eta_probe_epochs, probe_aux_size, num_w1_days)
precip_eta_result = measure_estimate(
    'ERA5-Land downscaling', 'eta continuation with IICT', run_precip_eta_probe, eta_scale,
    f'SRCNN eta, tau induced by max>150 mm, {num_w1_days} tail days, 150 epochs, lambda=1, omega=30',
    f'{precip_eta_probe_epochs} eta epoch with {probe_aux_size} auxiliary fields and {num_w1_days} W1 fields',
    precip_eta_probe_epochs,
    notes=f'Initialized from {precip_eta_init_source}; scale uses field-passes; no checkpoint saved.',
)


## Flow Matching Training And Sampling Probes

The Flow Matching probes use the U-Net settings from `notebooks/ERA5Land-DGM.ipynb` and sample paths documented in `notebooks/ERA5Land-DGM-Plot.ipynb`: `UNet(image_channels=1, n_channels=16, ch_mults=(1,2,2,4), is_attn=(False,False,True,True), n_blocks=1)`, Adam with learning rate $3\times10^{-4}$, LR FM trained on 25 years for 80 epochs, HR FM trained on 0.5 years for 200 epochs, 9044 generated samples, 100 ODE steps for LR samples, and 200 ODE steps for HR samples. The probe trains only a few optimizer steps and samples only a few ODE steps, then scales by optimizer sample-passes or generated-sample ODE-step passes.

In [5]:

from flow_matching.path import AffineProbPath
from flow_matching.path.scheduler import CondOTScheduler

image_channels = 1
n_channels = 16
ch_mults = (1, 2, 2, 4)
is_attn = (False, False, True, True)
n_blocks = 1
fm_lr = 3e-4

def get_hires_dgm_train(num_years):
    train_count = int((num_years / 25) * len(tp_trim_numpy))
    dgm_train = np.clip(tp_trim_numpy[:train_count], 1e0, None)
    dgm_train = np.log(dgm_train)
    train_mean = dgm_train.mean()
    train_std = dgm_train.std()
    return ((dgm_train - train_mean) / train_std).astype(np.float32)

def get_lores_dgm_train(num_years):
    train_count = int((num_years / 25) * len(tp_trim_ds_numpy))
    dgm_train = np.clip(tp_trim_ds_numpy[:train_count], 1e0, None)
    dgm_train = np.log(dgm_train)
    train_mean = dgm_train.mean()
    train_std = dgm_train.std()
    return ((dgm_train - train_mean) / train_std).astype(np.float32)

def build_fm_model():
    model = UNet(image_channels=image_channels, n_channels=n_channels, ch_mults=ch_mults, is_attn=is_attn, n_blocks=n_blocks)
    return maybe_data_parallel(model).to(DEVICE)

def train_fm_steps(train_array, n_steps, train_batch_size):
    global_seed(40)
    model = build_fm_model()
    path = AffineProbPath(scheduler=CondOTScheduler())
    optimizer = torch.optim.Adam(model.parameters(), lr=fm_lr)
    cfm_loss = nn.MSELoss()
    tensor = torch.tensor(train_array, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(tensor), batch_size=train_batch_size, shuffle=True, num_workers=0)
    steps_done = 0
    model.train()
    while steps_done < n_steps:
        for (x_1,) in loader:
            optimizer.zero_grad()
            x_1 = x_1.to(DEVICE)
            x_0 = torch.randn_like(x_1)
            t = torch.rand(x_1.shape[0], device=DEVICE)
            path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)
            loss = cfm_loss(model(path_sample.x_t, path_sample.t), path_sample.dx_t)
            loss.backward()
            optimizer.step()
            steps_done += 1
            if steps_done >= n_steps:
                break
    return model

def dopri5_step(v, x, t, h):
    k1 = v(x, t)
    k2 = v(x + h * (1 / 5) * k1, t + h / 5)
    k3 = v(x + h * (3 / 40 * k1 + 9 / 40 * k2), t + 3 * h / 10)
    k4 = v(x + h * (44 / 45 * k1 - 56 / 15 * k2 + 32 / 9 * k3), t + 4 * h / 5)
    k5 = v(x + h * (19372 / 6561 * k1 - 25360 / 2187 * k2 + 64448 / 6561 * k3 - 212 / 729 * k4), t + 8 * h / 9)
    k6 = v(x + h * (9017 / 3168 * k1 - 355 / 33 * k2 + 46732 / 5247 * k3 + 49 / 176 * k4 - 5103 / 18656 * k5), t + h)
    return x + h * (35 / 384 * k1 + 500 / 1113 * k3 + 125 / 192 * k4 - 2187 / 6784 * k5 + 11 / 84 * k6)

def fm_sampling_probe(model, n_steps, nsamples, batch_size, image_size):
    H, W, C = image_size
    h = 1.0 / max(n_steps, 1)
    model.eval()
    checksum = 0.0
    with torch.no_grad():
        for start in range(0, nsamples, batch_size):
            current = min(batch_size, nsamples - start)
            x = torch.randn(current, C, H, W, device=DEVICE)
            t = torch.zeros(current, device=DEVICE)
            for _ in range(n_steps):
                x = dopri5_step(model, x, t, h)
                t += h
            checksum += float(x.mean().detach().cpu())
    return checksum

def eta_pass_probe(model, nsamples, batch_size):
    model.eval()
    tensor = torch.randn(nsamples, 1, 8, 16, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=False, num_workers=0)
    checksum = 0.0
    with torch.no_grad():
        for (x,) in loader:
            out = model(x.to(DEVICE))
            checksum += float(out.mean().detach().cpu())
    return checksum

hr_fm_train = get_hires_dgm_train(0.5)
lr_fm_train = get_lores_dgm_train(25)
fm_train_probe_steps = 1
hr_fm_probe_batch = 32 if torch.cuda.is_available() else 1
lr_fm_probe_batch = 128 if torch.cuda.is_available() else 16

hr_fm_model = measure_estimate(
    'Flow Matching', 'HR FM training', train_fm_steps,
    (200 * len(hr_fm_train)) / (fm_train_probe_steps * hr_fm_probe_batch),
    'HR U-Net FM, 0.5-year HR data, 200 epochs, paper batch=128, Adam lr=3e-4',
    f'{fm_train_probe_steps} optimizer step(s), probe batch={hr_fm_probe_batch}, HR shape=(80,160)',
    hr_fm_train, fm_train_probe_steps, hr_fm_probe_batch,
    notes='Scale uses optimizer sample-passes; no checkpoint saved.',
)

lr_fm_model = measure_estimate(
    'Flow Matching', 'LR FM training', train_fm_steps,
    (80 * len(lr_fm_train)) / (fm_train_probe_steps * lr_fm_probe_batch),
    'LR U-Net FM, 25-year LR data, 80 epochs, paper batch=128, Adam lr=3e-4',
    f'{fm_train_probe_steps} optimizer step(s), probe batch={lr_fm_probe_batch}, LR shape=(8,16)',
    lr_fm_train, fm_train_probe_steps, lr_fm_probe_batch,
    notes='Scale uses optimizer sample-passes; no checkpoint saved.',
)

# If a training probe failed, create fresh models so sampling probes can still run.
if hr_fm_model is None:
    hr_fm_model = build_fm_model()
if lr_fm_model is None:
    lr_fm_model = build_fm_model()

hr_sample_probe_steps = 1
lr_sample_probe_steps = 5
hr_sample_probe_n = 32 if torch.cuda.is_available() else 1
lr_sample_probe_n = 512 if torch.cuda.is_available() else 4
hr_sample_probe_batch = hr_sample_probe_n
lr_sample_probe_batch = lr_sample_probe_n
nsamples_full = 9044

measure_estimate(
    'Flow Matching', 'HR FM sampling', fm_sampling_probe,
    (nsamples_full * 200) / (hr_sample_probe_n * hr_sample_probe_steps),
    'HR Dormand-Prince 5(4) sampling, 9044 samples, 200 ODE steps, HR shape=(80,160)',
    f'{hr_sample_probe_n} sample(s), {hr_sample_probe_steps} ODE step(s), batch={hr_sample_probe_batch}',
    hr_fm_model, hr_sample_probe_steps, hr_sample_probe_n, hr_sample_probe_batch, (80, 160, 1),
    notes='Scale uses generated-sample ODE-step passes; samples are not stored or saved.',
)
measure_estimate(
    'Flow Matching', 'LR FM sampling', fm_sampling_probe,
    (nsamples_full * 100) / (lr_sample_probe_n * lr_sample_probe_steps),
    'LR Dormand-Prince 5(4) sampling, 9044 samples, 100 ODE steps, LR shape=(8,16)',
    f'{lr_sample_probe_n} sample(s), {lr_sample_probe_steps} ODE step(s), batch={lr_sample_probe_batch}',
    lr_fm_model, lr_sample_probe_steps, lr_sample_probe_n, lr_sample_probe_batch, (8, 16, 1),
    notes='Scale uses generated-sample ODE-step passes; samples are not stored or saved.',
)

eta_pass_model = precip_eta_model if 'precip_eta_model' in globals() else build_srcnn()
eta_pass_probe_n = 128 if torch.cuda.is_available() else 16
measure_estimate(
    'Flow Matching', 'eta pass-through of LR samples', eta_pass_probe,
    nsamples_full / eta_pass_probe_n,
    'Pass 9044 LR generated samples through trained eta SRCNN, batch=256 in plotting notebook',
    f'{eta_pass_probe_n} synthetic LR sample(s), SRCNN eta pass-through',
    eta_pass_model, eta_pass_probe_n, 256,
    notes='No generated LR samples are loaded or saved; random LR tensors only time the eta map pass-through.',
)


0.15402455627918243

## GEVD Eta Timing Probe

The GEVD probe follows `notebooks/ERA5Land-EVD.ipynb`: the hypothesized heavy-tailed reference is a fitted/truncated GEVD law, the eta training uses $\tau=0.95$, 350 quantile values, 150 epochs, $\lambda=1$, and $\omega=1$. The probe uses the same SRCNN architecture and 0.5-year supervised split as vanilla ERA5-Land downscaling, but only one eta epoch on the bounded auxiliary set.

In [6]:

from scipy.integrate import quad
from scipy.stats import genextreme


def fit_gev_with_cutoff(data, cutoff):
    filtered_data = data[data <= cutoff]
    shape, loc, scale = genextreme.fit(filtered_data)
    scale = scale + 4
    normalization_constant = quad(
        lambda t: genextreme.pdf(t, shape, loc=loc, scale=scale),
        -np.inf,
        cutoff,
    )[0]
    return {
        'shape': shape,
        'location': loc,
        'scale': scale,
        'cutoff': cutoff,
        'c': normalization_constant,
    }

fit_results = fit_gev_with_cutoff(max_values, cutoff=np.ceil(float(np.max(max_values))) + 20)
tau_gevd = 0.95
num_w1_days_gevd = 350
quantiles_gevd = torch.linspace(tau_gevd, 1, num_w1_days_gevd)
Qy_gevd = lambda q: genextreme.ppf(q * fit_results['c'], fit_results['shape'], fit_results['location'], fit_results['scale'])
w1_truemax_gevd = torch.tensor([Qy_gevd(float(q)) for q in quantiles_gevd], dtype=torch.float32)
probe_sorted_indices_gevd = np.argsort(np.max(tp_probe_numpy, axis=(1, 2)))
w1_truedays_gevd_probe = probe_sorted_indices_gevd[-num_w1_days_gevd:]

gevd_eta_model, gevd_eta_init_source = load_precip_mse_or_fresh()

def run_gevd_eta_probe(num_epochs):
    model = deepcopy(gevd_eta_model).to(DEVICE)
    return train_precip_eta_no_save(
        model, probe_test_loader, train_input, train_target, tp_probe_ds_numpy,
        w1_truemax_gevd, w1_truedays_gevd_probe, num_epochs, 3e-4, 1.0, 1, True,
        seed=43, device=DEVICE, keep_best_by_w1=True,
    )

gevd_eta_probe_epochs = 1
gevd_scale = eta_field_pass_work(150, n_full_fields, num_w1_days_gevd) / eta_field_pass_work(gevd_eta_probe_epochs, probe_aux_size, num_w1_days_gevd)
measure_estimate(
    'GEVD prior downscaling', 'GEVD eta continuation', run_gevd_eta_probe, gevd_scale,
    f'SRCNN eta with fitted/truncated GEVD prior, tau=0.95, {num_w1_days_gevd} quantiles, 150 epochs, lambda=1, omega=1',
    f'{gevd_eta_probe_epochs} eta epoch with {probe_aux_size} auxiliary fields and {num_w1_days_gevd} GEVD W1 fields',
    gevd_eta_probe_epochs,
    notes=(
        f'Initialized from {gevd_eta_init_source}; fitted GEVD shape={fit_results["shape"]:.3f}, '
        f'loc={fit_results["location"]:.3f}, scale={fit_results["scale"]:.3f}; no checkpoint saved.'
    ),
)

gevd_setup = pd.DataFrame([{
    'tau': tau_gevd,
    'num_quantiles': num_w1_days_gevd,
    'shape_kappa': fit_results['shape'],
    'location_zeta': fit_results['location'],
    'scale_sigma': fit_results['scale'],
    'cutoff': fit_results['cutoff'],
    'normalization_c': fit_results['c'],
}])
display(gevd_setup)


,tau,num_quantiles,shape_kappa,location_zeta,scale_sigma,cutoff,normalization_c
0,0.95,350,-0.178688,25.077246,25.928067,258.0,0.995304


## Estimated Runtime And Memory Summary

The table below is the notebook-visible result. `estimated_full_time` is the proportional estimate for the full paper-scale setting listed in `paper_setting`. `peak_vram_allocated_GB` and `peak_vram_reserved_GB` come from the probe run; when no CUDA GPU is visible, VRAM fields are `NaN` and the process RSS columns document the observed host-memory footprint of the executed probes.

In [7]:

runtime_summary_df = pd.DataFrame(runtime_rows)
ordered_columns = [
    'experiment', 'component', 'status', 'paper_setting', 'timed_probe',
    'probe_seconds', 'probe_time', 'scale_factor', 'estimated_full_seconds', 'estimated_full_time',
    'peak_vram_allocated_GB', 'peak_vram_reserved_GB', 'process_rss_before_GB', 'process_rss_after_GB',
    'device', 'notes',
]
runtime_summary_df = runtime_summary_df[ordered_columns]
for col in ['probe_seconds', 'scale_factor', 'estimated_full_seconds', 'peak_vram_allocated_GB', 'peak_vram_reserved_GB', 'process_rss_before_GB', 'process_rss_after_GB']:
    runtime_summary_df[col] = runtime_summary_df[col].astype(float).round(4)
display(runtime_summary_df)

compact_summary_df = runtime_summary_df[[
    'experiment', 'component', 'status', 'estimated_full_time',
    'peak_vram_allocated_GB', 'peak_vram_reserved_GB', 'process_rss_after_GB', 'timed_probe',
]].copy()
display(compact_summary_df)

save_block_df = pd.DataFrame(SAVE_BLOCK_EVENTS) if SAVE_BLOCK_EVENTS else pd.DataFrame([{'api': 'none', 'target': 'No save calls attempted.'}])
display(save_block_df)


,experiment,component,status,paper_setting,timed_probe,probe_seconds,probe_time,scale_factor,estimated_full_seconds,estimated_full_time,peak_vram_allocated_GB,peak_vram_reserved_GB,process_rss_before_GB,process_rss_after_GB,device,notes
0,Toy 2D-to-1D,MSE baseline training,ok,"FCNN 256x3, psilu, Adam lr=1e-4, 3000 iterations, batch=100",10 iterations with full toy supervised set,5.1738,0:00:05.17,300.0000,1552.1289,0:25:52.13,0.0273,0.0430,2.0707,3.9831,cuda,Scale uses iteration count; no checkpoint saved.
1,Toy 2D-to-1D,eta pretraining,ok,"Gaussian random-field pretrain, 1000 iterations, n_grid=50, grid_step=2",10 pretrain iterations,0.1803,0:00:00.18,100.0000,18.0328,0:00:18.03,0.0291,0.0469,3.9831,3.9912,cuda,Probe uses same 50x50 pretraining grid as the notebook.
2,Toy 2D-to-1D,eta continuation,ok,"eta continuation, lambda=1, omega=100, 185 quantiles, 3000 iterations",10 eta iterations after a one-step warm-start pretrain,0.7183,0:00:00.72,300.0000,215.4892,0:03:35.49,0.5232,0.5625,3.9912,4.1084,cuda,Scale uses iteration count with the full toy auxiliary grid.
3,Toy 2D-to-2D,MSE baseline training,ok,"FCNN 256x3, psilu, Adam defaults, 3000 iterations, batch=100",10 iterations with full toy supervised set,0.1450,0:00:00.15,300.0000,43.5129,0:00:43.51,0.0273,0.0430,4.1084,4.1084,cuda,Scale uses iteration count; no checkpoint saved.
4,Toy 2D-to-2D,eta pretraining,ok,"Two-channel Gaussian/Fourier pretrain, 1000 iterations, n_grid=50, grid_step=2",10 pretrain iterations,0.1607,0:00:00.16,100.0000,16.0679,0:00:16.07,0.0295,0.0469,4.1084,4.1085,cuda,Probe uses the same two-channel pretraining target as the notebook.
5,Toy 2D-to-2D,eta continuation,ok,"eta continuation, lambda=1, omega=100, 167 quantiles, 3000 iterations",10 eta iterations after a one-step warm-start pretrain,0.6088,0:00:00.61,300.0000,182.6430,0:03:02.64,0.5390,0.6836,4.1085,4.1094,cuda,Scale uses iteration count with the full toy auxiliary grid.
6,ERA5-Land downscaling,MSE baseline training,ok,"SRCNN, 0.5-year supervised split, 500 epochs, batch=64, Adam lr=3e-4, StepLR(200, 0.2)",1 epoch over 180 supervised fields plus 768 auxiliary eval fields,0.6758,0:00:00.68,4864.9789,3287.5423,0:54:47.54,0.3413,0.5664,4.1095,4.5006,cuda,Scale uses supervised plus auxiliary field-passes; no checkpoint saved.
7,ERA5-Land downscaling,eta continuation with IICT,ok,"SRCNN eta, tau induced by max>150 mm, 102 tail days, 150 epochs, lambda=1, omega=30",1 eta epoch with 768 auxiliary fields and 102 W1 fields,0.8062,0:00:00.81,774.4466,624.3338,0:10:24.33,0.8985,1.4902,4.5006,4.6464,cuda,Initialized from loaded srcnn-mse-0.5yr-10ds.pth; scale uses field-passes; no checkpoint saved.
8,Flow Matching,HR FM training,ok,"HR U-Net FM, 0.5-year HR data, 200 epochs, paper batch=128, Adam lr=3e-4","1 optimizer step(s), probe batch=32, HR shape=(80,160)",0.7839,0:00:00.78,1125.0000,881.9398,0:14:41.94,0.3078,0.3906,4.6466,5.1148,cuda,Scale uses optimizer sample-passes; no checkpoint saved.
9,Flow Matching,LR FM training,ok,"LR U-Net FM, 25-year LR data, 80 epochs, paper batch=128, Adam lr=3e-4","1 optimizer step(s), probe batch=128, LR shape=(8,16)",0.3441,0:00:00.34,5652.5000,1944.9498,0:32:24.95,0.2575,0.3613,5.1148,5.1418,cuda,Scale uses optimizer sample-passes; no checkpoint saved.


,experiment,component,status,estimated_full_time,peak_vram_allocated_GB,peak_vram_reserved_GB,process_rss_after_GB,timed_probe
0,Toy 2D-to-1D,MSE baseline training,ok,0:25:52.13,0.0273,0.0430,3.9831,10 iterations with full toy supervised set
1,Toy 2D-to-1D,eta pretraining,ok,0:00:18.03,0.0291,0.0469,3.9912,10 pretrain iterations
2,Toy 2D-to-1D,eta continuation,ok,0:03:35.49,0.5232,0.5625,4.1084,10 eta iterations after a one-step warm-start pretrain
3,Toy 2D-to-2D,MSE baseline training,ok,0:00:43.51,0.0273,0.0430,4.1084,10 iterations with full toy supervised set
4,Toy 2D-to-2D,eta pretraining,ok,0:00:16.07,0.0295,0.0469,4.1085,10 pretrain iterations
5,Toy 2D-to-2D,eta continuation,ok,0:03:02.64,0.5390,0.6836,4.1094,10 eta iterations after a one-step warm-start pretrain
6,ERA5-Land downscaling,MSE baseline training,ok,0:54:47.54,0.3413,0.5664,4.5006,1 epoch over 180 supervised fields plus 768 auxiliary eval fields
7,ERA5-Land downscaling,eta continuation with IICT,ok,0:10:24.33,0.8985,1.4902,4.6464,1 eta epoch with 768 auxiliary fields and 102 W1 fields
8,Flow Matching,HR FM training,ok,0:14:41.94,0.3078,0.3906,5.1148,"1 optimizer step(s), probe batch=32, HR shape=(80,160)"
9,Flow Matching,LR FM training,ok,0:32:24.95,0.2575,0.3613,5.1418,"1 optimizer step(s), probe batch=128, LR shape=(8,16)"


,api,target
0,none,No save calls attempted.


# Revision Documentation Notes

This notebook estimates computational overhead without producing new artifacts. It reports tested hardware, bounded probe runtimes, proportional full-run estimates, and probe memory usage for the toy examples, ERA5-Land SRCNN downscaling, Flow Matching training/sampling, eta pass-through of generated LR samples, and the GEVD-prior eta run. The timing cells intentionally avoid `torch.save`, `np.save`, and figure generation.